In [1]:
import copy

import datasets
from datasets import load_dataset, DatasetDict, Dataset, ClassLabel
import pandas as pd
from peft import LoraConfig, LoraModel, get_peft_model
from transformers import BertForSequenceClassification, BertTokenizer
from torch.utils.data import DataLoader
from typing import Collection
from tqdm import tqdm


import transformers
import torch
import torch.nn as nn

import optuna

2026-01-28 12:54:35.853685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769604876.043869      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769604876.101963      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769604876.545867      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769604876.545910      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769604876.545913      55 computation_placer.cc:177] computation placer alr

In [2]:
torch.manual_seed(42)

author_to_label = {
    "blok": 0,
    "cvetaeva": 1,
    "pasternak": 2
}
transformers.logging.set_verbosity_error()

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(device)

cuda:0


In [3]:
def preprocess_data(data: pd.DataFrame):
    processed_data = data.copy()
    processed_data["text_length"] = processed_data["text"].str.split().str.len()
    processed_data = processed_data[processed_data["text_length"] < 1000]
    processed_data["label"] = processed_data["label"].apply(lambda x: author_to_label[x])
    return processed_data

def df_to_dataset(df: pd.DataFrame) -> DatasetDict:
    data = Dataset.from_pandas(df)
    data = data.cast_column("label", ClassLabel(num_classes=3))
    data = data.shuffle()
    data = data.train_test_split(test_size=0.2, stratify_by_column="label")
    return data

def tokenize_text(data: Dataset, tokenizer: BertTokenizer):
    return tokenizer(
        data["text"],
        padding=True,
        truncation=False,
    )

In [4]:
model_name = "cointegrated/rubert-tiny2"

model = BertForSequenceClassification.from_pretrained(model_name, num_labels=3).to(device)
tokenizer = BertTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
data = pd.read_csv("/kaggle/input/dataset/output.csv")
data = preprocess_data(data)
dataset = df_to_dataset(data)
dataset = dataset.map(lambda x: tokenize_text(x, tokenizer), batched=True, batch_size=64)
dataset.set_format(type="torch")

train_data_loader = DataLoader(dataset["train"], batch_size=16, shuffle=False)
test_data_loader = DataLoader(dataset["test"], batch_size=16, shuffle=False)

Casting the dataset:   0%|          | 0/1174 [00:00<?, ? examples/s]

Map:   0%|          | 0/939 [00:00<?, ? examples/s]

Map:   0%|          | 0/235 [00:00<?, ? examples/s]

In [6]:
from sklearn.metrics import f1_score


def finetune(study, model, n=200):
    def training_loop(trial):
        lora_config = LoraConfig(
            r=trial.suggest_int("r", 4, 16),
            lora_alpha=trial.suggest_int("r", 4, 16) * trial.suggest_int("alpha_multiplicator", 1, 4),
            target_modules=["query", "value", "key", "output.dense"],
            lora_dropout=trial.suggest_float("lora_dropout", 0, 0.15),
            task_type='SEQ_CLS'
        )
        lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
        
        lora_model = get_peft_model(model, lora_config)
        optimizer = torch.optim.AdamW(lora_model.parameters(), lr=lr)
        
        worse_epoch_count = 0
        for epoch in range(n):
            print(f"Epoch #{epoch + 1}")
            print(f"-" * 30)
            if worse_epoch_count == 0:
                old_model = copy.deepcopy(lora_model)
            if epoch > 0:
                prev_f1 = f1
            else:
                prev_f1 = float("inf")
            total_train_loss = 0
            total_test_loss = 0
            accuracy = 0
            
            lora_model.train()
            for batch in tqdm(train_data_loader):
                optimizer.zero_grad()
                outputs = lora_model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["label"].to(device)
                )
                loss = outputs.loss
                total_train_loss += loss.item()
                loss.backward()
                optimizer.step()
            
            print(f'Avg train loss: {total_train_loss / len(train_data_loader)}')
            
            lora_model.eval()
            ys, ps = [], []
            for batch in test_data_loader:
                test_outputs = lora_model(
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["label"].to(device)
                )
                total_test_loss += test_outputs.loss.item()
                predictions = torch.argmax(test_outputs.logits, dim=1)
                ys.append(batch["label"])
                ps.append(predictions)
            y = torch.concatenate(ys).to("cpu")
            p = torch.concatenate(ps).to("cpu")
            
            accuracy = (p == y).sum().item() / len(y)
            f1 = f1_score(y, p, average="macro")
    
            total_test_loss /= len(test_data_loader)
            
            print(f'Avg test loss: {total_test_loss}')
            print(f'Test accuracy: {accuracy}')
            print(f"Macro f1: {f1}")
            if f1 < prev_f1:
                worse_epoch_count += 1
                if worse_epoch_count == 2:
                    print("Early stopping...")
                    return prev_f1
            else:
                worse_epoch_count = 0
            print(f"-" * 30)
        return f1

    study.optimize(training_loop, n_trials=15)
    return study.best_params

In [7]:
study = optuna.create_study(study_name="study", direction="minimize")
best_params = finetune(study, model)
print(best_params)

[I 2026-01-28 12:54:57,342] A new study created in memory with name: study


Epoch #1
------------------------------


100%|██████████| 59/59 [00:12<00:00,  4.84it/s]


Avg train loss: 1.0713243130910195
Avg test loss: 1.0359572887420654
Test accuracy: 0.5063829787234042
Macro f1: 0.224105461393597
------------------------------
Epoch #2
------------------------------


100%|██████████| 59/59 [00:11<00:00,  5.10it/s]


...

Avg train loss: 0.7700478828559487
Avg test loss: 0.7852850914001465
Test accuracy: 0.6425531914893617
Macro f1: 0.5920838340511507
------------------------------
Epoch #2
------------------------------


100%|██████████| 59/59 [00:11<00:00,  5.08it/s]


Avg train loss: 0.7249825637219316


[I 2026-01-28 14:08:40,339] Trial 14 finished with value: 0.5920838340511507 and parameters: {'r': 6, 'alpha_multiplicator': 1, 'lora_dropout': 0.09785628775546144, 'lr': 3.470423001875749e-05}. Best is trial 11 with value: 0.5882762778857104.


Avg test loss: 0.7602112213770549
Test accuracy: 0.6425531914893617
Macro f1: 0.5881871831842052
Early stopping...
{'r': 6, 'alpha_multiplicator': 1, 'lora_dropout': 0.10769100694439593, 'lr': 3.444187587702093e-05}
